# Phase 4 · Resolve modular feature profiles

V4 assembles a profile from reusable modules in a defined order: core, pathogen group, host group, host taxon, and pair context. Resolution records both applicable features and reasons for features that do not apply.

This phase resolves configuration only. Scoring decisions are kept in the decision ledger and use their own scope-specific cache keys.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository')
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id, log_step
RUN_ID = configured_run_id(PROJECT_ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('04_feature_profiles', RUN_ID)
print('Project root:', ROOT, '| log:', LOG_PATH)

2026-11-02 12:12:20,879 INFO Notebook session started at 2026-11-02T12:12:20.879726+00:00


2026-11-02 12:12:20,880 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4


2026-11-02 12:12:20,880 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/04_feature_profiles.log


Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4 | log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/04_feature_profiles.log


## Step 1 · Load and review the feature catalogue

In [2]:
import yaml, pandas as pd
from IPython.display import display
from pathogen_atlas_v4.features import resolve_profile
CATALOGUE_PATH = ROOT / 'configs/features.yaml'
CATALOGUE = yaml.safe_load(CATALOGUE_PATH.read_text())
print('Catalogue:', CATALOGUE_PATH)
print('Catalogue version:', CATALOGUE.get('catalogue_version'))
display(pd.DataFrame([{'module': key, 'feature_count': len(features)} for key, features in CATALOGUE['modules'].items()]))
LOG.info('Feature catalogue modules=%d', len(CATALOGUE['modules']))

Catalogue: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/configs/features.yaml
Catalogue version: feature-catalogue-v1


,module,feature_count
0,core,1
1,pathogen_group:bacteria,0
2,pathogen_group:fungi,0
3,pathogen_group:viruses,0
4,host_group:human,0
5,host_group:livestock,0
6,host_group:crop,0
7,pair_context:disease,0


2026-11-02 12:12:21,306 INFO Feature catalogue modules=8


## Step 2 · Resolve one profile for each pilot host context

In [3]:
import yaml
CONFIG = yaml.safe_load((ROOT / 'configs/pilot_v1.yaml').read_text())
TAXONOMY_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1]
hosts = pd.read_parquet(TAXONOMY_ROOT / 'hosts.parquet')
PATHOGEN_GROUPS = list(CONFIG['run'].get('pathogen_groups', []))
if not PATHOGEN_GROUPS:
    import json
    resolved_path = ROOT / 'data/v4/runs' / CONFIG['run']['run_id'] / 'resolved_config.json'
    resolved = json.loads(resolved_path.read_text())
    selected_ids = set(map(str, resolved['pathogen_ids']))
    run_pathogens = pd.read_parquet(ROOT / 'data/v4/runs' / CONFIG['run']['run_id'] / 'taxonomy/pathogens.parquet')
    PATHOGEN_GROUPS = sorted(run_pathogens.loc[run_pathogens.pathogen_id.astype(str).isin(selected_ids), 'pathogen_group'].astype(str).unique())
PROFILE_VERSION = 'host-aware-profile-v1'
profiles = []
for host in hosts.itertuples(index=False):
    for pathogen_group in PATHOGEN_GROUPS:
        profile = resolve_profile(CATALOGUE, pathogen_group=pathogen_group,
            host_group=host.host_group, host_taxon_id=host.taxon_id,
            relationship_context='disease', profile_version=PROFILE_VERSION)
        profiles.append({'host_id': host.host_id, 'host': host.accepted_name,
                         'host_group': host.host_group, 'pathogen_group': pathogen_group,
                         'resolved_profile': profile})
profile_summary = pd.DataFrame([{'host_id': p['host_id'], 'host': p['host'],
    'host_group': p['host_group'], 'pathogen_group': p['pathogen_group'],
    'applicable_features': len(p['resolved_profile']['ordered_features']),
    'not_applicable_features': len(p['resolved_profile']['not_applicable']),
    'profile_hash': p['resolved_profile']['resolved_profile_hash']} for p in profiles])
display(profile_summary)
LOG.info('Resolved %d host × pathogen-group profiles', len(profiles))

,host_id,host,host_group,pathogen_group,applicable_features,not_applicable_features,profile_hash
0,ncbi:3847,Glycine max,crop,bacteria,1,0,65c54d01920c5adc7021d2ed7fb54c5238537ef88fdab4...
1,ncbi:3847,Glycine max,crop,fungi,1,0,fad831f2dd0aa46044f7c32e23a463197d075ceba91b7a...
2,ncbi:3847,Glycine max,crop,viruses,1,0,2ddac89d9b9f455d5240d608ab008cea577285c9c5d85a...
3,ncbi:4530,Oryza sativa,crop,bacteria,1,0,9595f03477faff7e4ef17bd771f5d716fd8e3914930d30...
4,ncbi:4530,Oryza sativa,crop,fungi,1,0,d29961dcbcef6bb6493e1544ffd9c8a9db1419e50d2ff0...
5,ncbi:4530,Oryza sativa,crop,viruses,1,0,0c77347f7d67a8f38fef9d43299bd7a6d6c28b6ad732aa...
6,ncbi:4565,Triticum aestivum,crop,bacteria,1,0,d9f4b54b09c032f54bfad739ba99cec5e9ce69b7aab266...
7,ncbi:4565,Triticum aestivum,crop,fungi,1,0,2f58962407a70a1b4310ab892fba8f2dfc14f612673878...
8,ncbi:4565,Triticum aestivum,crop,viruses,1,0,0c3bbd321ec2a8181fdb62cc3855753e8485c2cccbced9...
9,ncbi:4577,Zea mays,crop,bacteria,1,0,544fbffc6c0c86861f429ad3d6394fa244c3690cb666fd...


2026-11-02 12:12:21,338 INFO Resolved 27 host × pathogen-group profiles


## Step 3 · Inspect feature scopes and exclusions

In [4]:
selected_profile = next(p['resolved_profile'] for p in profiles
                       if p['host_id']=='ncbi:9913' and p['pathogen_group']=='bacteria')
selected_features = pd.DataFrame(selected_profile['ordered_features'])
not_applicable = pd.DataFrame(selected_profile['not_applicable'])
print('Example profile:', selected_profile['profile_id'], selected_profile['profile_version'])
print('Included feature scopes:')
display(selected_features.groupby(['module','scope']).size().rename('features').to_frame())
print('Not-applicable reasons:')
display(not_applicable.head(25))
LOG.info('Example resolved profile hash=%s included=%d excluded=%d', selected_profile['resolved_profile_hash'], len(selected_features), len(not_applicable))

Example profile: host-aware-default host-aware-profile-v1
Included feature scopes:


,,features
module,scope,
core,pathogen_host_pair,1


Not-applicable reasons:


""


2026-11-02 12:12:21,344 INFO Example resolved profile hash=4e3f65b4fb0f6548632c153be437c4c9e7ba6e7d1bd8d978431380074f78e869 included=1 excluded=0


## Step 4 · Save resolved profiles for the run

In [5]:
import json
PROFILE_DIR = ROOT / 'data/v4/runs' / RUN_ID / 'feature_profiles'
PROFILE_DIR.mkdir(parents=True, exist_ok=True)
PROFILE_PATH = PROFILE_DIR / f'{selected_profile["resolved_profile_hash"]}.json'
PROFILE_PATH.write_text(json.dumps(selected_profile, indent=2, ensure_ascii=False) + '\n')
print('Saved profile:', PROFILE_PATH)
LOG.info('Saved profile hash=%s path=%s', selected_profile['resolved_profile_hash'], PROFILE_PATH)

2026-11-02 12:12:21,348 INFO Saved profile hash=4e3f65b4fb0f6548632c153be437c4c9e7ba6e7d1bd8d978431380074f78e869 path=/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/pilot-v1/feature_profiles/4e3f65b4fb0f6548632c153be437c4c9e7ba6e7d1bd8d978431380074f78e869.json


Saved profile: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/pilot-v1/feature_profiles/4e3f65b4fb0f6548632c153be437c4c9e7ba6e7d1bd8d978431380074f78e869.json


## Step 5 · Scoring handoff

In [6]:
print('Profile resolution is local and deterministic; no model calls were made.')
print('The scoring stage should generate pathogen-, host-, or pair-scoped tasks from these ordered feature definitions.')
LOG.info('Feature scoring handoff ready; model calls=0')

2026-11-02 12:12:21,351 INFO Feature scoring handoff ready; model calls=0


Profile resolution is local and deterministic; no model calls were made.
The scoring stage should generate pathogen-, host-, or pair-scoped tasks from these ordered feature definitions.
